In [1]:
pip install google-cloud-bigquery pandas pyarrow db-dtypes requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 163.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 214.7 MB/s  0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.80.0
    Uninstalling grpcio-1.80.0:
      Successfully uninstalled grpcio-1.80.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [google-cloud-bigquery]
Note: you may need to restart the kernel to use updated packages.


**Підключення до BigQuery**

In [2]:
import os, sys, json
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

PROJECT_ID = "project-nbu" # Замініть на свій!
LOCATION   = "EU"

creds = None
if os.environ.get("GCP_SA_KEY"):
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("Проєкт підключено успішно:", client.project)

Проєкт підключено успішно: project-nbu


In [3]:
# --- ЗАВДАННЯ 2.1: Створення датасетів nbu_raw і nbu_dwh ---

for dataset_name in ["nbu_raw", "nbu_dwh"]:
    ds = bigquery.Dataset(f"{PROJECT_ID}.{dataset_name}")
    ds.location = "EU"
    # exists_ok=True гарантирует, что код не упадет, если датасеты уже созданы
    client.create_dataset(ds, exists_ok=True)
    print(f"Датасет {dataset_name} готов к работе в регионе EU.")

Датасет nbu_raw готов к работе в регионе EU.
Датасет nbu_dwh готов к работе в регионе EU.


**Завдання 2.2.** Створіть таблицю nbu_raw.raw_rates з такою схемою:
Колонка	Тип	Що зберігає
ingested_at	TIMESTAMP	момент запису в сховище
business_date	DATE	дата, ЗА яку взято курс
request_url	STRING	повний URL запиту
payload	STRING	один об’єкт відповіді як JSON-текст
payload_hash	STRING	MD5 від payload

In [4]:
# --- ЗАВДАННЯ 2.2: Створення партиційованої таблиці raw_rates ---

TABLE_ID = f"{PROJECT_ID}.nbu_raw.raw_rates"

# Описуємо схему таблиці відповідно до вимог завдання
schema = [
    bigquery.SchemaField("ingested_at", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("business_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("request_url", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("payload", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("payload_hash", "STRING", mode="REQUIRED"),
]

# Створюємо об'єкт таблиці
table = bigquery.Table(TABLE_ID, schema=schema)

# Налаштовуємо щоденне партиціювання за полем business_date
table.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field="business_date"
)

# Створюємо таблицю в BigQuery (exists_ok=True захищає від помилок при повторному запуску)
client.create_table(table, exists_ok=True)

print(f"✅ Таблицю {TABLE_ID} успішно створено з партиціюванням за business_date.")

✅ Таблицю project-nbu.nbu_raw.raw_rates успішно створено з партиціюванням за business_date.


In [5]:
import requests

URL = "https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange?json"

# Робимо запит до API з тайм-аутом та перевіркою статусу
resp = requests.get(URL, timeout=30)
resp.raise_for_status()

data = resp.json()  # Список словників
df_raw = pd.DataFrame(data)

print(f"Успішно завантажено {len(df_raw)} валют з API НБУ.")
print("Приклад першого рядка:")
print(data[0])

Успішно завантажено 45 валют з API НБУ.
Приклад першого рядка:
{'r030': 12, 'txt': 'Алжирський динар', 'rate': 0.33495, 'cc': 'DZD', 'exchangedate': '31.08.2026', 'special': None}


In [6]:
table_ref = f"{PROJECT_ID}.nbu_raw.exchange_raw"

# Стратегія: Тільки дописуємо нові дані і ніколи їх не змінюємо
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND
)

# Завантажуємо DataFrame у BigQuery
job = client.load_table_from_dataframe(df_raw, table_ref, job_config=job_config)
job.result()  # Очікуємо завершення завантаження

print(f"✅ Дані успішно дозаписано в таблицю {table_ref}.")

✅ Дані успішно дозаписано в таблицю project-nbu.nbu_raw.exchange_raw.


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


*Задайте партиціювання за business_date:*

In [7]:
table = bigquery.Table(TABLE_ID, schema=schema)
table.time_partitioning = bigquery.TimePartitioning(field="business_date")
client.create_table(table, exists_ok=True)

Table(TableReference(DatasetReference('project-nbu', 'nbu_raw'), 'raw_rates'))

**Завдання 2.3.** Заберіть із API курс на сьогодні та виведіть кількість отриманих валют і перший елемент відповіді.

In [8]:
# --- ЗАВДАННЯ 2.3: Отримання курсів валют з API НБУ ---
import requests

URL = "https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange?json"

# Робимо запит з тайм-аутом та перевіркою на помилки сервера
resp = requests.get(URL, timeout=30)
resp.raise_for_status()

# Зберігаємо відповідь у вигляді списку словників
data = resp.json()

# Виводимо кількість отриманих валют і перший елемент
print(f"Кількість отриманих валют: {len(data)}")
print("Перший елемент відповіді:")
print(data[0])


Кількість отриманих валют: 45
Перший елемент відповіді:
{'r030': 12, 'txt': 'Алжирський динар', 'rate': 0.33495, 'cc': 'DZD', 'exchangedate': '31.08.2026', 'special': None}


**Завдання 2.4.** Перетворіть відповідь на рядки таблиці: один об’єкт відповіді = один рядок. Колонку payload заповніть текстом усього об’єкта (json.dumps), а payload_hash — його MD5-хешем.

In [9]:
# --- ЗАВДАННЯ 2.4: Преобразование данных в формат таблицы raw_rates ---
import hashlib
import json
from datetime import datetime

rows = []
# Фиксируем момент записи и URL запроса для всех строк текущего запуска
ingested_at = datetime.utcnow()
request_url = "https://bank.gov.ua"

for item in data:
    # 1. Сериализуем объект валюты в JSON-строку
    payload = json.dumps(item, ensure_ascii=False, sort_keys=True)

    # 2. Вычисляем MD5-хеш от полученной строки
    payload_hash = hashlib.md5(payload.encode("utf-8")).hexdigest()

    # 3. Преобразуем дату курса "ДД.ММ.РРРР" в "РРРР-ММ-ДД" для BigQuery DATE
    raw_date = item["exchangedate"]  # например, "17.08.2026"
    business_date = datetime.strptime(raw_date, "%d.%m.%Y").date()

    # Собираем строку для будущей таблицы
    rows.append(
        {
            "ingested_at": ingested_at,
            "business_date": business_date,
            "request_url": request_url,
            "payload": payload,
            "payload_hash": payload_hash,
        }
    )

# Создаем итоговый DataFrame
df_bronze = pd.DataFrame(rows)

# Проверка: выводим форму датафрейма и первые 2 строки
print(f"Формат DataFrame: {df_bronze.shape} (строк, колонок)")
print("\nПример первых двух строк:")
print(df_bronze.head(2))

Формат DataFrame: (45, 5) (строк, колонок)

Пример первых двух строк:
                 ingested_at business_date          request_url  \
0 2026-08-29 19:41:38.457135    2026-08-31  https://bank.gov.ua   
1 2026-08-29 19:41:38.457135    2026-08-31  https://bank.gov.ua   

                                             payload  \
0  {"cc": "DZD", "exchangedate": "31.08.2026", "r...   
1  {"cc": "AUD", "exchangedate": "31.08.2026", "r...   

                       payload_hash  
0  e312d961d82f1722a1782e37b1a567c8  
1  73dc491854b5b8c326b2612aa04d2a2f  


**Завдання 2.5.** Запишіть DataFrame у nbu_raw.raw_rates у режимі WRITE_APPEND

In [10]:
# --- ЗАВДАННЯ 2.5: Запись DataFrame в BigQuery (Режим APPEND) ---

TABLE_ID = f"{PROJECT_ID}.nbu_raw.raw_rates"

# Настраиваем конфигурацию загрузки, передавая созданную ранее schema
cfg = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition="WRITE_APPEND"
)

# Запускаем загрузку DataFrame в таблицу BigQuery
job = client.load_table_from_dataframe(df_bronze, TABLE_ID, job_config=cfg)
job.result()  # Ожидаем завершения загрузки

print(f"✅ Данные успешно записаны в таблицу {TABLE_ID}!")

✅ Данные успешно записаны в таблицу project-nbu.nbu_raw.raw_rates!


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


**Завдання 2.6.** Виконайте перевірочний запит і виведіть кількість рядків за кожною business_date, відсортовану за датою:

In [11]:
# --- ЗАВДАННЯ 2.6: Проверочный SQL-запрос ---

# Записываем текст SQL-запроса, подставляя ваш PROJECT_ID
query = f"""
SELECT business_date, COUNT(*) AS rows_cnt, COUNT(DISTINCT payload_hash) AS uniq_cnt
FROM `{PROJECT_ID}.nbu_raw.raw_rates`
GROUP BY business_date
ORDER BY business_date
"""

# Выполняем запрос и переводим результат в Pandas DataFrame
df_check = client.query(query).to_dataframe()

# Выводим результат на экран
print(df_check)

  business_date  rows_cnt  uniq_cnt
0    2026-08-25        90        45
1    2026-08-31        45        45


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [12]:
# --- ЗАВДАННЯ 2.7: Объяснение эффекта дублирования данных ---

# Объяснение:
# При повторном запуске общая сумма строк rows_cnt удваивается, так как таблица 
# работает в режиме WRITE_APPEND (строки просто дописываются в конец). 
# При этом количество уникальных хешей uniq_cnt остается прежним, так как 
# данные из API НБУ за сегодняшний день не изменились, и повторно загруженные 
# строки полностью идентичны первым (их MD5-хеши совпадают).

query = f"""
SELECT business_date, COUNT(*) AS rows_cnt, COUNT(DISTINCT payload_hash) AS uniq_cnt
FROM `{PROJECT_ID}.nbu_raw.raw_rates`
GROUP BY business_date
ORDER BY business_date
"""

df_check = client.query(query).to_dataframe()
print(df_check)

  business_date  rows_cnt  uniq_cnt
0    2026-08-25        90        45
1    2026-08-31        45        45


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
